In [1]:
from pathlib import Path
import json
import subprocess
import sys

import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from IPython.display import Markdown, display

src_path = Path.cwd() / "src"
if src_path.exists() and str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import dm_spikes as dm

In [2]:
M = 2.6e6
D = 8.5e3
rho0_iso = 0.0062
sigma_v = 100.0
m = 8.963e-56  # Msun
observable_sigma_v = 3.221e-74  # pc^3 / yr
bh_age = 1e10  # yr
R_S = dm.schwarzschild_radius(M)

In [3]:
archivo_a_cargar = "results/gs_numerical_reconstruction.npz"
data = np.load(archivo_a_cargar, allow_pickle=True)

r_iso = data["r_iso"]
rho_iso = data["rho_iso"]
gamma_values = data["gamma_values"]
r_cusp_grid = data["r_cusp_grid"]
rho_cusp_grid = data["rho_cusp_grid"]
g_gamma_grid = data["g_gamma_grid"]
dgdr_grid = data["dgdr_grid"]
rho_spike_grid = data["rho_spike_grid"]
R_core_values = data["R_core_values"]
R_sp_values = data["R_sp_values"]

print("Perfiles cargados desde archivo.")
print("r_iso:", r_iso.shape)
print("rho_iso:", rho_iso.shape)
print("gamma_values:", gamma_values.shape)
print("r_cusp_grid:", r_cusp_grid.shape)
print("rho_cusp_grid:", rho_cusp_grid.shape)
print("g_gamma_grid:", g_gamma_grid.shape)
print("dgdr_grid:", dgdr_grid.shape)
print("rho_spike_grid:", rho_spike_grid.shape)
print("R_core_values:", R_core_values.shape)
print("R_sp_values:", R_sp_values.shape)

Perfiles cargados desde archivo.
r_iso: (300,)
rho_iso: (300,)
gamma_values: (195,)
r_cusp_grid: (195, 300)
rho_cusp_grid: (195, 300)
g_gamma_grid: (195, 300)
dgdr_grid: (195, 300)
rho_spike_grid: (195, 300)
R_core_values: (195,)
R_sp_values: (195,)


In [4]:
g_floor = 1e-12
fit_output_dir = Path("results") / "capture_fit_latest"

cmd = [
    sys.executable,
    "scripts/fit_capture_factor.py",
    "--input", archivo_a_cargar,
    "--output-dir", str(fit_output_dir),
    "--g-floor", str(g_floor),
    "--M", str(M),
]
completed = subprocess.run(cmd, check=True, capture_output=True, text=True)
report = json.loads(completed.stdout)

fit_parameters = np.load(report["npz"], allow_pickle=True)
summary_text = Path(report["summary"]).read_text(encoding="utf-8")

display(Markdown("### Ajuste global de $g_\\gamma(r)$"))
print(summary_text)
print("\nArchivos generados:")
print("-", report["npz"])
print("-", report["summary"])
print("-", report["figures"])

report

### Ajuste global de $g_\gamma(r)$

Ajuste global de g_gamma(r)

Formula final: g_fit(r,gamma) = A(gamma) x**B(gamma), x = 1 - 4 R_S/r.
Modelo seleccionado: simple_log_refined
Grados seleccionados: A=0, B=2
Dominio gamma: [0.01, 1.95]
Dominio r/R_S usado: [4.001, 3.204347e+08]

Coeficientes Chebyshev log A: [0.015111398609]
Coeficientes Chebyshev log B: [0.881504408394 0.046003868094 0.008627176525]

Metricas finales: RMSE=3.31095320e-02, MAE=2.58885945e-02, max_abs=1.21500499e-01, log_RMSE=5.33825228e-02
Comparacion directa: RMSE=2.11718953e-02, log_RMSE=2.71536157e-01
Comparacion log: RMSE=3.31095320e-02, log_RMSE=5.33825228e-02
B_i medio por filas: 2.4084858; referencia Gondolo-Silk B=3.

Advertencias:
- A(gamma) es una normalizacion efectiva y puede reflejar rho_R, R_sp o gamma_sp, no solo fisica de captura.


Archivos generados:
- results\capture_fit_latest\capture_factor_global_fit.npz
- results\capture_fit_latest\capture_factor_global_fit_summary.txt
- results\capture_fit_latest\figures


{'npz': 'results\\capture_fit_latest\\capture_factor_global_fit.npz',
 'summary': 'results\\capture_fit_latest\\capture_factor_global_fit_summary.txt',
 'figures': 'results\\capture_fit_latest\\figures',
 'model_name': 'simple_log_refined',
 'formula': 'A(gamma) x**B(gamma)',
 'selected': {'deg_A': 0,
  'deg_B': 2,
  'cv_log_rmse': 0.053063743463419,
  'folds': 195},
 'coeffs_logA': [0.015111398608519715],
 'coeffs_logB': [0.8815044083943646,
  0.04600386809356462,
  0.008627176525459907],
 'coeffs_C': None,
 'metrics': {'rmse': 0.03310953203625534,
  'mae': 0.025888594479126222,
  'max_abs': 0.12150049867750301,
  'log_rmse': 0.05338252278280823},
 'direct_metrics': {'rmse': 0.02117189533668565,
  'mae': 0.015533211853918554,
  'max_abs': 0.09170290896379363,
  'log_rmse': 0.27153615669396636},
 'log_metrics': {'rmse': 0.03310953203625534,
  'mae': 0.025888594479126222,
  'max_abs': 0.12150049867750301,
  'log_rmse': 0.05338252278280823},
 'warnings': ['A(gamma) es una normalizacion e

In [5]:
def g_fit_directo(r, gamma, R_S=R_S, fit_parameters=fit_parameters):
    """Evalua directamente el ajuste global de g_gamma(r).

    Usa los coeficientes Chebyshev guardados en fit_parameters y acepta
    escalares o arreglos mediante broadcasting.
    """
    r = np.asarray(r, dtype=float)
    gamma = np.asarray(gamma, dtype=float)

    gamma_min, gamma_max = fit_parameters["gamma_domain"]
    gamma_hat = 2.0 * (gamma - gamma_min) / (gamma_max - gamma_min) - 1.0

    logA = np.polynomial.chebyshev.chebval(gamma_hat, fit_parameters["cheb_coeffs_logA"])
    logB = np.polynomial.chebyshev.chebval(gamma_hat, fit_parameters["cheb_coeffs_logB"])
    A = np.exp(logA)
    B = np.exp(logB)

    x = 1.0 - 4.0 * R_S / r
    x, A, B = np.broadcast_arrays(x, A, B)
    g_fit = np.zeros_like(x, dtype=float)

    valid = np.isfinite(x) & np.isfinite(A) & np.isfinite(B) & (x > 0.0)
    g_fit[valid] = A[valid] * x[valid] ** B[valid]

    coeffs_C = fit_parameters["cheb_coeffs_C"]
    if coeffs_C.size:
        C = np.polynomial.chebyshev.chebval(gamma_hat, coeffs_C)
        C = np.broadcast_to(C, g_fit.shape)
        g_fit[valid] *= np.exp(C[valid] * (1.0 - x[valid]))

    return g_fit


g_fit_directo(r_cusp_grid[0, :5], gamma_values[0])

array([4.24723118e-09, 1.24431220e-03, 5.81235464e-03, 1.39608490e-02,
       2.55350100e-02])

In [ ]:
x_numerico, y_numerico, z_numerico, color_numerico = [], [], [], []
x_ajuste, y_ajuste, z_ajuste = [], [], []
max_puntos_por_curva = 600

for i, gamma in enumerate(gamma_values):
    r = np.asarray(r_cusp_grid[i], dtype=float)
    g = np.asarray(g_gamma_grid[i], dtype=float)
    indices = np.flatnonzero(np.isfinite(r) & np.isfinite(g) & (r > 4.0 * R_S))

    if indices.size > max_puntos_por_curva:
        seleccion = np.linspace(0, indices.size - 1, max_puntos_por_curva, dtype=int)
        indices = indices[seleccion]

    x = np.log10(r[indices] / R_S)
    g_ajustado = g_fit_directo(r[indices], gamma)
    altura_gamma = np.full(x.shape, gamma)

    x_numerico.extend(x.tolist() + [None])
    y_numerico.extend(g[indices].tolist() + [None])
    z_numerico.extend(altura_gamma.tolist() + [None])
    color_numerico.extend([float(gamma)] * (x.size + 1))

    x_ajuste.extend(x.tolist() + [None])
    y_ajuste.extend(g_ajustado.tolist() + [None])
    z_ajuste.extend(altura_gamma.tolist() + [None])

fig = go.Figure()

fig.add_trace(
    go.Scatter3d(
        x=x_numerico,
        y=y_numerico,
        z=z_numerico,
        mode="lines",
        name="Reconstruccion numerica",
        line=dict(
            color=color_numerico,
            colorscale="Viridis",
            width=3,
            colorbar=dict(title="gamma"),
        ),
        opacity=0.65,
        hovertemplate=(
            "Numerico<br>log10(r/R_S) = %{x:.4f}<br>"
            "g = %{y:.6g}<br>gamma = %{z:.4f}<extra></extra>"
        ),
    )
)

fig.add_trace(
    go.Scatter3d(
        x=x_ajuste,
        y=y_ajuste,
        z=z_ajuste,
        mode="lines",
        name="Ajuste global",
        line=dict(color="#D62728", width=4),
        opacity=0.9,
        hovertemplate=(
            "Ajuste<br>log10(r/R_S) = %{x:.4f}<br>"
            "g_fit = %{y:.6g}<br>gamma = %{z:.4f}<extra></extra>"
        ),
    )
)

fig.update_layout(
    title="Reconstruccion numerica y ajuste global",
    width=1000,
    height=750,
    scene=dict(
        xaxis_title="log10(r / R_S)",
        yaxis_title="g_gamma(r)",
        zaxis_title="gamma",
        aspectmode="manual",
        aspectratio=dict(x=1.5, y=1.0, z=1.0),
        camera=dict(eye=dict(x=1.5, y=-1.5, z=1.0)),
        dragmode="orbit",
    ),
    legend=dict(x=0.01, y=0.99),
    margin=dict(l=0, r=0, b=0, t=45),
)

fig.show()